# Simulating Morphometric Traits Across Evolutionary Time

This notebook is a reproducible, linear re-implementation of an R/Shiny application
that simulates how a 3D landmark-based (geometric morphometric) skull shape evolves
along a phylogenetic tree, reconstructs ancestral shapes, and visualizes the result
in morphospace over geological time.

**What the workflow does, end to end:**

1. Loads (or, in this demo, simulates) 3D landmark coordinates for a set of taxa,
   read in Morphologika format.
2. Performs a Generalized Procrustes Analysis (GPA) to remove position, rotation and
   scale, then runs a Principal Component Analysis (PCA) on the aligned shapes to
   build a low-dimensional "shape space".
3. Optionally groups landmarks into biologically meaningful **modules** (e.g.
   "Snout", "Braincase") and builds a block-structured evolutionary-rate matrix
   ($\Sigma$) that lets landmarks within the same module evolve in a correlated way.
4. Simulates trait evolution for every tip of a phylogenetic tree under either
   **Brownian Motion (BM)** or an **Ornstein-Uhlenbeck (OU)** process in PC space.
5. Reconstructs ancestral states at every internal node with `Rphylopars`.
6. Slices the reconstructed evolutionary history at regular time intervals and
   back-projects each slice from PC space into landmark (skull) space.
7. Re-aligns (GPA) all time slices together and visualizes the expansion of
   morphospace through time, the phylomorphospace tree, and individual 3D
   wireframe skulls.
8. Runs a Mantel test comparing morphological (Procrustes) distance to
   phylogenetic distance, as a simple diagnostic of phylogenetic signal.

The original implementation was an interactive Shiny app. This notebook keeps all of the original analysis
functions unchanged in substance, but replaces the Shiny reactive app
with a **Parameters** cell and a linear sequence of
code cells, so the whole analysis can be re-run top-to-bottom and produce the same figures/tables every time, in order for the pipeline to be reproducible.


**Source.** This notebook reproduces the analysis pipeline (`app.R`) published
in the GitHub repository
[`ekazasi/Morphometric-traits-over-time`](https://github.com/ekazasi/Morphometric-traits-over-time),
which accompanies the report *Kazasi, E. (2026). "Simulating morphometric
traits across evolutionary time: a novel pipeline." Bioinformatics.* The
repository's own `README.md` file and `environment.yml` file are used to denote
package versions and to load the same example dataset (94 primate cranial
specimens, 31 landmarks) used in the trial run.

| Package | Used for |
| --- | --- |
| `geomorph` | GPA superimposition (`gpagen`), Morphologika I/O, 2D-array conversion |
| `mvMORPH` | Simulating BM / OU trait evolution along the tree (`mvSIM`) |
| `phytools` | Tree utilities (`nodeHeights`, `branching.times`, `compute.brtime`) |
| `Rphylopars` | Maximum-likelihood ancestral state reconstruction (`phylopars`) |
| `vegan` | Mantel test (`mantel`) |
| `viridis` | Colour palettes for time gradients |
| `plotly` | Interactive 3D wireframe skull rendering |
| `ggplot2`, `dplyr` | General data wrangling / plotting support |
| `ggtree` | Optional tree-plotting extensions |
| `BiocManager` | Installs `ggtree` from Bioconductor |
| `RRPP` | `geomorph` dependency (permutational ANOVA backend) |

**Exact, pinned environment (recommended).** The source repository
([`ekazasi/Morphometric-traits-over-time`](https://github.com/ekazasi/Morphometric-traits-over-time))
contains a conda `environment.yml` that reproduces the exact versions the
original analysis was run with. Follow the instructions in the `README.md` to replicate the suggested environment.

*Ps. This script has been written in order to be a interactive app, but for the purposes of this exercise I transfered the code in a jupyter notebook*

-> The jupyter notebook is a plain top-to-bottom script split into cells. Change a parameter → re-run the cells below it manually (Kernel → Restart & Run All). There's no automatic recomputation and no running server, as in the shiny app.

## Mathematical background

**Shape space (GPA + PCA).** Generalized Procrustes Analysis removes
translation, rotation and scale from a set of landmark configurations, leaving
only *shape*. PCA on the resulting aligned (Procrustes) coordinates gives an
orthogonal, variance-ranked "shape space": a configuration's PC score is
$$ s = (x - \bar{x})\,V $$
where $x$ is the vectorized aligned landmark configuration, $\bar{x}$ the
consensus (mean) shape, and $V$ the matrix of retained eigenvectors
(loadings).

The simulation employs two primary mathematical frameworks:

1. **Brownian Motion (BM).** Modeled as:
$$ dX(t) = \sigma\, dW(t) $$
where shape change is a random walk with a rate $\sigma$. This represents
neutral evolution (genetic drift): morphological changes occur randomly,
without a specific directional pressure, so variance among lineages grows
linearly with elapsed time.

2. **Ornstein-Uhlenbeck (OU).** Modeled as:
$$ dX(t) = a(\theta - X(t))\,dt + \sigma\, dW(t) $$
where $a$ represents the strength of selection pulling the trait towards an
optimum shape $\theta$, on top of the same random (Brownian) component
$\sigma\,dW(t)$.

Between two tree nodes $t_p < t_c$ with states $x_p, x_c$, the OU expected
trajectory at an intermediate time $t$ is interpolated as
$$ \hat{x}(t) = x_p + \frac{1-e^{-a (t-t_p)}}{1-e^{-a (t_c-t_p)}}\,(x_c - x_p) $$
which reduces to plain linear interpolation, $\hat x(t) = x_p + \tfrac{t-t_p}{t_c-t_p}(x_c-x_p)$,
in the BM case ($a \to 0$).

**Ancestral state reconstruction.** Internal node states are estimated by
maximum likelihood under the fitted BM/OU model (`Rphylopars::phylopars`),
using the observed tip PC scores and the tree topology/branch lengths.

**Mantel test.** Tests whether morphological (Procrustes) distance between
taxa, $D_{morph}$, correlates with phylogenetic path distance, $D_{phylo}$:
$$ r = \text{cor}\big(\text{vec}(D_{morph}),\ \text{vec}(D_{phylo})\big) $$
with significance assessed by permuting one matrix's labels many times
(999 permutations here) and comparing the observed $r$ to the permutation
null distribution.



## 1. Install and load packages

In [ ]:
options(repos = c(CRAN = "https://cloud.r-project.org"))

packages <- c("geomorph", "mvMORPH", "phytools", "vegan", "Rphylopars",
              "viridis", "plotly", "ggplot2", "dplyr")

missing_pkgs <- packages[!(packages %in% installed.packages()[, "Package"])]
if (length(missing_pkgs)) {
  message("Installing missing packages: ", paste(missing_pkgs, collapse = ", "))
  install.packages(missing_pkgs, dependencies = TRUE)
} else {
  message("All CRAN packages are already installed.")
}

if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if (!requireNamespace("ggtree", quietly = TRUE)) BiocManager::install("ggtree")

invisible(lapply(c(packages, "ggtree"), library, character.only = TRUE))

set.seed(42)  # reproducibility for the synthetic-data / simulation steps below


## 2. Core analysis functions

The functions below are unchanged in substance from the original application;
only the Shiny reactive wiring around them has been removed. 

### 2.1 Geometry utilities

In [ ]:
# Convert a row-wise shape matrix into a geomorph-compatible 3D array
# [Landmarks, Dimensions, Specimens]
reshape_to_array <- function(sim_data, n_land) {
  n_sp <- nrow(sim_data)
  array(t(sim_data), dim = c(n_land, 3, n_sp))
}


### 2.2 Trait-evolution simulation and ancestral reconstruction

In [ ]:
# Simulate tip trait scores in PC space under BM or OU dynamics.
# BM:  X(t) = X(0) + epsilon,  Var[epsilon] proportional to branch length.
# OU:  dX_t = alpha(theta - X_t) dt + Sigma dW_t.
simulate_tip_scores <- function(tree, root_state, sigma_matrix, target_state = NULL,
                                 model = "BM", alpha_val = NULL, pc_labels = NULL) {
  if (model == "BM") {
    tip_scores <- mvSIM(tree, model = "BM1",
                         param = list(theta = root_state, sigma = sigma_matrix))
  } else {
    n_pcs <- length(root_state)
    alpha_matrix <- if (is.matrix(alpha_val)) alpha_val else diag(alpha_val, n_pcs)
    tip_scores <- mvSIM(tree, model = "OU1",
                         param = list(theta = target_state, sigma = sigma_matrix,
                                      alpha = alpha_matrix))
    # Anchor the mean to the ancestor consensus (root_state) rather than the
    # stationary optimum, preserving relative variation.
    root_offset <- root_state - colMeans(tip_scores)
    tip_scores <- sweep(tip_scores, 2, root_offset, "+")
  }

  rownames(tip_scores) <- tree$tip.label
  if (!is.null(pc_labels) && length(pc_labels) == ncol(tip_scores)) {
    colnames(tip_scores) <- pc_labels
  }
  tip_scores
}

# Reconstruct internal node states with maximum likelihood using Rphylopars.
estimate_ancestral_scores <- function(tree, tip_scores, model = "BM") {
  trait_data <- data.frame(species = tree$tip.label, tip_scores, check.names = FALSE)
  recon_fit <- phylopars(
    trait_data = trait_data,
    tree = tree,
    model = if (model == "OU") "OU" else "BM",
    pheno_error = FALSE,
    phylo_correlated = TRUE,
    pheno_correlated = TRUE
  )

  reconstructed_scores <- recon_fit$anc_recon
  n_tips <- Ntip(tree)
  full_history <- matrix(NA, nrow = n_tips + tree$Nnode, ncol = ncol(reconstructed_scores))
  rownames(full_history) <- as.character(seq_len(n_tips + tree$Nnode))
  colnames(full_history) <- colnames(reconstructed_scores)
  full_history[as.character(seq_len(n_tips)), ] <- reconstructed_scores[tree$tip.label, , drop = FALSE]
  internal_node_ids <- as.character((n_tips + 1):(n_tips + tree$Nnode))
  full_history[internal_node_ids, ] <- reconstructed_scores[internal_node_ids, , drop = FALSE]

  list(fit = recon_fit, full_history = full_history)
}

# Run simulation + ancestral reconstruction and return the full tip+node history.
simulate_and_reconstruct_history <- function(tree, root_state, sigma_matrix, target_state = NULL,
                                              model = "BM", alpha_val = NULL, pc_labels = NULL) {
  tip_scores <- simulate_tip_scores(tree, root_state, sigma_matrix, target_state = target_state,
                                     model = model, alpha_val = alpha_val, pc_labels = pc_labels)
  ancestral_estimate <- estimate_ancestral_scores(tree, tip_scores, model = model)
  ancestral_estimate$full_history[as.character(Ntip(tree) + 1), ] <- root_state

  list(tip_scores = tip_scores,
       reconstruction = ancestral_estimate$fit,
       full_history = ancestral_estimate$full_history)
}


### 2.3 Sampling shapes at fixed time slices

In [ ]:
# Sample branch states at a geological time slice (Ma) by interpolating
# between parent and child node states in PC space.
sample_tree_at_time <- function(tree, full_data, target_time, alpha_val = NULL, model = NULL) {
  edges <- tree$edge
  heights <- nodeHeights(tree)

  slice_coords <- matrix(NA, nrow = 0, ncol = ncol(full_data))
  lineage_ids <- c()

  for (e in seq_len(nrow(edges))) {
    start_h <- heights[e, 1]
    end_h <- heights[e, 2]

    if (target_time >= start_h && target_time <= end_h) {
      parent_node <- edges[e, 1]
      child_node <- edges[e, 2]
      p_coords <- full_data[as.character(parent_node), ]
      c_coords <- full_data[as.character(child_node), ]
      dt_slice <- target_time - start_h
      branch_duration <- end_h - start_h

      if (model == "OU") {
        local_pull <- if (branch_duration == 0) 1 else
          (1 - exp(-alpha_val * dt_slice)) / (1 - exp(-alpha_val * branch_duration))
        interp <- p_coords + local_pull * (c_coords - p_coords)
      } else {
        prop <- dt_slice / branch_duration
        interp <- p_coords + prop * (c_coords - p_coords)
      }

      slice_coords <- rbind(slice_coords, interp)
      lineage_ids <- c(lineage_ids, child_node)
    }
  }

  rownames(slice_coords) <- lineage_ids
  return(slice_coords)
}

# Generate time-ordered snapshot matrices across all requested intervals.
generate_all_snapshots <- function(tree, intervals, full_history, alpha_val = NULL, model = NULL) {
  snapshot_list <- list()
  for (i in seq_along(intervals)) {
    target_time <- intervals[i]
    snapshot_list[[paste0("Time_", target_time)]] <- sample_tree_at_time(
      tree, full_history, target_time, alpha_val = alpha_val, model = model
    )
    message(paste("Processed time slice:", target_time, "Ma"))
  }
  return(snapshot_list)
}


### 2.4 GPA / PCA alignment and diagnostics

In [ ]:
# Apply GPA jointly to all time-slice snapshots so any coordinate-space
# noise introduced by back-projection is removed before diagnostics/plots.
perform_final_gpa_on_snapshots <- function(snapshot_list, n_land) {
  snapshot_sizes <- vapply(snapshot_list, nrow, integer(1))
  all_snapshots <- do.call(rbind, snapshot_list)
  all_array <- array(t(all_snapshots), dim = c(n_land, 3, nrow(all_snapshots)))
  gpa_result <- gpagen(all_array, print.progress = FALSE)
  aligned_matrix <- two.d.array(gpa_result$coords)

  aligned_snapshots <- list()
  start_idx <- 1
  for (snapshot_name in names(snapshot_list)) {
    end_idx <- start_idx + snapshot_sizes[[snapshot_name]] - 1
    aligned_snapshots[[snapshot_name]] <- aligned_matrix[start_idx:end_idx, , drop = FALSE]
    rownames(aligned_snapshots[[snapshot_name]]) <- rownames(snapshot_list[[snapshot_name]])
    start_idx <- end_idx + 1
  }

  list(aligned_snapshots = aligned_snapshots, gpa_result = gpa_result)
}

# GPA-align a shape matrix (optionally plus a target configuration) and
# compute PCA coordinates, for diagnostic plotting.
align_shape_matrix_for_diagnostics <- function(shape_matrix, n_land, target_config = NULL) {
  all_points_matrix <- shape_matrix
  target_index <- NULL

  if (!is.null(target_config)) {
    all_points_matrix <- rbind(all_points_matrix, as.numeric(target_config))
    target_index <- nrow(all_points_matrix)
  }

  all_array <- array(t(all_points_matrix), dim = c(n_land, 3, nrow(all_points_matrix)))
  gpa_results <- gpagen(all_array, print.progress = FALSE)
  aligned_2d <- two.d.array(gpa_results$coords)
  pca_res <- prcomp(aligned_2d)

  list(aligned_matrix = aligned_2d, pca = pca_res, target_index = target_index)
}


### 2.5 Visualization functions

In [ ]:
# Plot the reconstructed tips and internal nodes as a connected tree in
# PCA morphospace (a "phylomorphospace").
plot_phylomorphospace <- function(tree, full_history, n_land, title = "Phylomorphospace",
                                   target_config = NULL) {
  diag_data <- align_shape_matrix_for_diagnostics(full_history, n_land, target_config = target_config)
  pca_scores <- diag_data$pca$x[seq_len(nrow(full_history)), 1:2, drop = FALSE]

  plot(pca_scores[, 1], pca_scores[, 2], type = "n",
       xlab = paste0("PC1 (", round(summary(diag_data$pca)$importance[2, 1] * 100, 1), "%)"),
       ylab = paste0("PC2 (", round(summary(diag_data$pca)$importance[2, 2] * 100, 1), "%)"),
       main = title)

  for (edge_index in seq_len(nrow(tree$edge))) {
    parent_node <- tree$edge[edge_index, 1]
    child_node <- tree$edge[edge_index, 2]
    segments(pca_scores[parent_node, 1], pca_scores[parent_node, 2],
             pca_scores[child_node, 1], pca_scores[child_node, 2],
             col = rgb(0.3, 0.3, 0.3, 0.6), lwd = 1.2)
  }

  tip_ids <- seq_len(Ntip(tree))
  node_ids <- (Ntip(tree) + 1):(Ntip(tree) + tree$Nnode)
  points(pca_scores[tip_ids, 1], pca_scores[tip_ids, 2], pch = 16, col = "black", cex = 0.8)
  points(pca_scores[node_ids, 1], pca_scores[node_ids, 2], pch = 21, bg = "steelblue",
         col = "white", cex = 1)

  if (!is.null(diag_data$target_index)) {
    target_pca <- diag_data$pca$x[diag_data$target_index, 1:2, drop = FALSE]
    points(target_pca[1, 1], target_pca[1, 2], col = "red", pch = 8, cex = 1.2, lwd = 2)
    text(target_pca[1, 1], target_pca[1, 2], labels = "optimum", pos = 3, col = "red", cex = 0.6)
  }
}

# Build an interactive 3D landmark + wireframe Plotly object for one skull.
build_wireframe_plotly <- function(coords, wireframe_links, title_text, point_color = "steelblue") {
  fig <- plot_ly(type = "scatter3d") |>
    add_trace(x = coords[, 1], y = coords[, 2], z = coords[, 3],
              mode = "markers+text", type = "scatter3d",
              marker = list(size = 3, color = point_color),
              text = as.character(seq_len(nrow(coords))),
              textposition = "top center",
              textfont = list(size = 10, color = "black"),
              hovertemplate = "Landmark %{text}<extra></extra>")

  if (!is.null(wireframe_links) && nrow(wireframe_links) > 0) {
    for (link_index in seq_len(nrow(wireframe_links))) {
      link_pair <- wireframe_links[link_index, ]
      fig <- fig |>
        add_trace(x = coords[link_pair, 1], y = coords[link_pair, 2], z = coords[link_pair, 3],
                  type = "scatter3d", mode = "lines",
                  line = list(color = "rgba(40, 40, 40, 0.7)", width = 4),
                  hoverinfo = "skip", showlegend = FALSE)
    }
  }

  plotly::layout(fig, title = list(text = title_text),
                  scene = list(xaxis = list(title = "X"), yaxis = list(title = "Y"),
                               zaxis = list(title = "Z"), aspectmode = "data"),
                  margin = list(l = 0, r = 0, b = 0, t = 40))
}

# Perform GPA -> PCA on all time-slice snapshots and render the temporal
# morphospace, colour-coded from old (dark) to recent (light).
analyze_rigorous_morphospace <- function(snapshot_list, n_land, title = "Shape Space Expansion",
                                          target_config = NULL, start_time_ma = 1, end_time_ma = 50) {
  all_points_matrix <- do.call(rbind, snapshot_list)

  if (!is.null(target_config)) {
    all_points_matrix <- rbind(all_points_matrix, as.numeric(target_config))
  }

  n_total_observations <- nrow(all_points_matrix)
  all_array <- array(t(all_points_matrix), dim = c(n_land, 3, n_total_observations))
  message("Performing Procrustes Alignment...")
  gpa_results <- gpagen(all_array, print.progress = FALSE)
  aligned_2d <- two.d.array(gpa_results$coords)
  pca_res <- prcomp(aligned_2d)

  if (!is.null(target_config)) {
    target_pca <- pca_res$x[n_total_observations, , drop = FALSE]
  }

  n_steps <- length(snapshot_list)
  colors_vec <- rev(viridis(n_steps))
  par(mar = c(5, 4, 4, 7))

  plot(pca_res$x[, 1], pca_res$x[, 2], type = "n",
       xlab = paste0("PC1 (", round(summary(pca_res)$importance[2, 1] * 100, 1), "%)"),
       ylab = paste0("PC2 (", round(summary(pca_res)$importance[2, 2] * 100, 1), "%)"),
       main = title)

  curr_row <- 1
  for (i in seq_along(snapshot_list)) {
    n_taxa <- nrow(snapshot_list[[i]])
    rows <- curr_row:(curr_row + n_taxa - 1)
    points(pca_res$x[rows, 1], pca_res$x[rows, 2], col = colors_vec[i], pch = 16, cex = 0.8)
    curr_row <- curr_row + n_taxa
  }

  if (!is.null(target_config)) {
    points(target_pca[1, 1], target_pca[1, 2], col = "red", pch = 8, cex = 1, lwd = 2)
    text(target_pca[1, 1], target_pca[1, 2], labels = "optimum", pos = 3, col = "red", cex = 0.4, font = 2)
  }

  u <- par("usr")
  bar_width <- (u[2] - u[1]) * 0.05
  left_edge <- u[2] + (u[2] - u[1]) * 0.02
  right_edge <- left_edge + bar_width
  y_steps <- seq(u[3], u[4], length.out = n_steps + 1)
  for (i in seq_len(n_steps)) {
    rect(left_edge, y_steps[i], right_edge, y_steps[i + 1], col = colors_vec[i], border = NA, xpd = TRUE)
  }
  rect(left_edge, u[3], right_edge, u[4], border = "black", xpd = TRUE)

  xr <- (max(pca_res$x[, 1]) + (max(pca_res$x[, 1]) - min(pca_res$x[, 1])) * 0.15)
  yb <- min(pca_res$x[, 2]); yt <- max(pca_res$x[, 2])
  xl <- (max(pca_res$x[, 1]) + (max(pca_res$x[, 1]) - min(pca_res$x[, 1])) * 0.05)
  text(x = xr, y = yb, labels = paste0(start_time_ma, " Ma"), pos = 4, cex = 0.8, xpd = TRUE)
  text(x = xr, y = yt, labels = paste0(end_time_ma, " Ma"), pos = 4, cex = 0.8, xpd = TRUE)
  text(x = (xl + xr) / 2, y = yt + (yt - yb) * 0.03, labels = "Time", pos = 3, cex = 0.9, font = 2, xpd = TRUE)
}


### 2.6 Data I/O

In [ ]:
# Read a Morphologika-format file and return coordinates plus metadata.
# Use this with your own data: morph_data <- load_morphologika_data("my_file.txt")
load_morphologika_data <- function(file_path) {
  all_skulls_local <- read.morphologika(file_path)
  coords_data_local <- all_skulls_local$coords

  reference_lines <- readLines(file_path, warn = FALSE)
  trimmed_reference_lines <- trimws(reference_lines)
  names_start <- match("[names]", trimmed_reference_lines)
  labels_start <- match("[labels]", trimmed_reference_lines)

  name_lines <- trimmed_reference_lines[(names_start + 1):(labels_start - 1)]
  name_lines <- name_lines[name_lines != ""]
  specimen_taxa_local <- vapply(name_lines, function(line) {
    parts <- strsplit(line, "\\s+")[[1]]
    paste(parts[1:min(2, length(parts))], collapse = " ")
  }, character(1))

  list(all_skulls = all_skulls_local, coords_data = coords_data_local,
       specimen_taxa = specimen_taxa_local, unique_taxa = sort(unique(specimen_taxa_local)),
       n_land = dim(coords_data_local)[1], p_dim = dim(coords_data_local)[2],
       n_vars = dim(coords_data_local)[1] * dim(coords_data_local)[2])
}

# Consensus landmark configuration for one selected taxon: align its
# specimens with gpagen and use the resulting consensus shape.
compute_taxon_consensus <- function(morph_data, taxon_name) {
  selected_idx <- which(morph_data$specimen_taxa == taxon_name)
  gpagen(morph_data$coords_data[, , selected_idx, drop = FALSE], print.progress = FALSE)$consensus
}


### 2.7 Tree calibration and PC-space / module construction

In [ ]:
# Apply optional node-age overrides and rebuild a calibrated branch-time tree.
build_calibrated_tree <- function(tree, calibration_text = NULL) {
  node_ids <- as.character((Ntip(tree) + 1):(Ntip(tree) + tree$Nnode))
  default_ages <- branching.times(tree)
  node_ages <- as.numeric(default_ages[node_ids])
  names(node_ages) <- node_ids

  overrides <- numeric(0)
  if (!is.null(calibration_text) && nzchar(trimws(calibration_text))) {
    lines <- trimws(unlist(strsplit(calibration_text, "\n")))
    lines <- lines[lines != ""]
    for (line in lines) {
      parts <- trimws(unlist(strsplit(line, "[:=,]")))
      parts <- parts[parts != ""]
      if (length(parts) == 2) {
        node_id <- as.character(as.integer(parts[1]))
        node_ages[node_id] <- as.numeric(parts[2])
        overrides[node_id] <- as.numeric(parts[2])
      }
    }
  }

  calibrated_tree <- compute.brtime(tree, method = node_ages, force.positive = FALSE)
  list(tree = calibrated_tree, node_ages = node_ages, applied_overrides = overrides)
}

# Build the PCA simulation space, module assignments, and Sigma (rate) matrix.
build_pc_simulation_space <- function(morph_data, ancestor_taxon, target_taxon, pc_mode,
                                       variance_threshold, requested_pcs, taxa_limit,
                                       module_text, sigma_diag, sigma_offdiag) {
  aligned_coords <- gpagen(morph_data$coords_data, print.progress = FALSE)$coords
  aligned_matrix <- two.d.array(aligned_coords)
  pca_result <- prcomp(aligned_matrix, center = TRUE, scale. = FALSE)

  explained_variance <- (pca_result$sdev ^ 2) / sum(pca_result$sdev ^ 2)
  cumulative_variance <- cumsum(explained_variance)
  threshold_pcs <- which(cumulative_variance >= variance_threshold)[1]
  max_pcs <- min(ncol(aligned_matrix), nrow(aligned_matrix) - 1, taxa_limit - 1)
  retained_pcs <- if (pc_mode == "manual") requested_pcs else threshold_pcs
  if (is.null(retained_pcs) || is.na(retained_pcs)) retained_pcs <- threshold_pcs
  retained_pcs <- min(retained_pcs, max_pcs)
  retained_indices <- seq_len(retained_pcs)
  retained_rotation <- pca_result$rotation[, retained_indices, drop = FALSE]

  # Build biologically motivated modules from landmark sets; PCs are later
  # grouped by dominant module loadings to induce block structure in Sigma.
  module_definitions <- if (is.null(module_text) || !nzchar(trimws(module_text))) {
    list(Global = seq_len(morph_data$n_land))
  } else {
    lines <- trimws(unlist(strsplit(module_text, "\n")))
    lines <- lines[lines != ""]
    modules <- list()
    assigned_landmarks <- integer(0)
    for (line in lines) {
      parts <- strsplit(line, ":", fixed = TRUE)[[1]]
      module_name <- trimws(parts[1])
      landmark_tokens <- trimws(unlist(strsplit(parts[2], "[,[:space:]]+")))
      landmark_tokens <- landmark_tokens[landmark_tokens != ""]
      landmark_ids <- suppressWarnings(as.integer(landmark_tokens))
      modules[[module_name]] <- sort(unique(landmark_ids))
      assigned_landmarks <- c(assigned_landmarks, landmark_ids)
    }
    unassigned <- setdiff(seq_len(morph_data$n_land), assigned_landmarks)
    if (length(unassigned) > 0) modules[["Unassigned"]] <- unassigned
    modules
  }

  module_names <- names(module_definitions)
  module_scores <- matrix(0, nrow = retained_pcs, ncol = length(module_definitions))
  for (module_index in seq_along(module_definitions)) {
    coord_ids <- unlist(lapply(module_definitions[[module_index]], function(landmark_id) {
      ((landmark_id - 1) * 3 + 1):(landmark_id * 3)
    }))
    module_scores[, module_index] <- colSums(abs(retained_rotation[coord_ids, , drop = FALSE]))
  }

  dominant_modules <- max.col(module_scores, ties.method = "first")
  pc_order <- order(dominant_modules, seq_len(retained_pcs))
  ordered_rotation <- retained_rotation[, pc_order, drop = FALSE]
  ordered_pc_indices <- retained_indices[pc_order]
  ordered_module_names <- module_names[dominant_modules[pc_order]]

  # Sigma is diagonal by default; within-module off-diagonal entries encode
  # correlated evolution (e.g. PC1 & PC2 both dominated by "Snout").
  sigma_matrix <- diag(sigma_diag, retained_pcs)
  for (i in seq_len(retained_pcs)) {
    for (j in seq_len(retained_pcs)) {
      if (i != j && ordered_module_names[i] == ordered_module_names[j]) sigma_matrix[i, j] <- sigma_offdiag
    }
  }
  pc_labels <- paste0("PC", ordered_pc_indices)
  rownames(sigma_matrix) <- pc_labels
  colnames(sigma_matrix) <- pc_labels

  ancestor_idx <- which(morph_data$specimen_taxa == ancestor_taxon)
  target_idx <- which(morph_data$specimen_taxa == target_taxon)
  ancestor_consensus <- apply(aligned_coords[, , ancestor_idx, drop = FALSE], c(1, 2), mean)
  target_consensus <- apply(aligned_coords[, , target_idx, drop = FALSE], c(1, 2), mean)
  ancestor_vector <- as.vector(t(ancestor_consensus))
  target_vector <- as.vector(t(target_consensus))

  pca_model <- list(aligned_coords = aligned_coords, aligned_matrix = aligned_matrix,
                     pca_result = pca_result, explained_variance = explained_variance,
                     cumulative_variance = cumulative_variance, threshold_pcs = threshold_pcs,
                     max_pcs = max_pcs, retained_pcs = retained_pcs, retained_indices = retained_indices,
                     center = pca_result$center, retained_rotation = retained_rotation)

  sigma_info <- list(sigma_matrix = sigma_matrix, ordered_rotation = ordered_rotation,
                      ordered_pc_indices = ordered_pc_indices, ordered_module_names = ordered_module_names,
                      pc_labels = pc_labels)

  # root/target states in PC space, plus objects needed to back-project
  # from retained PC space to landmark coordinate space.
  list(root_state = as.numeric((ancestor_vector - pca_model$center) %*% sigma_info$ordered_rotation),
       theta_state = as.numeric((target_vector - pca_model$center) %*% sigma_info$ordered_rotation),
       ancestor_vector = ancestor_vector, target_vector = target_vector, center = pca_model$center,
       rotation = sigma_info$ordered_rotation, sigma_matrix = sigma_info$sigma_matrix,
       pca_model = pca_model, module_definitions = module_definitions, sigma_info = sigma_info)
}

# Format calibrated node ages into a readable reference table.
format_node_age_reference <- function(tree, node_ages) {
  node_ids <- as.integer(names(node_ages))
  parent_lookup <- setNames(tree$edge[, 1], tree$edge[, 2])
  child_counts <- table(tree$edge[, 1])

  lines <- vapply(node_ids, function(node_id) {
    parent_id <- parent_lookup[as.character(node_id)]
    descendant_count <- if (as.character(node_id) %in% names(child_counts)) child_counts[[as.character(node_id)]] else 0
    parent_label <- if (is.na(parent_id)) "root" else as.character(parent_id)
    paste0("Node ", node_id, " | age = ", round(node_ages[as.character(node_id)], 3),
           " | parent = ", parent_label, " | child branches = ", descendant_count)
  }, character(1))

  paste(lines, collapse = "\n")
}


## 3. Load data

Replace this cell with `morph_data <- load_morphologika_data("path/to/your_file.txt")`
to use your own Morphologika file.

For a **runnable, self-contained demo**, the cell below synthesizes a small
`morph_data` object with the exact structure `load_morphologika_data()` would
return: 12 landmarks placed on an ellipsoid ("skull"), two taxa (an
`"Ancestor"` and a `"Target"` morph) with 6 specimens each, where the target's
mean shape is a fixed deformation of the ancestor's (simulating, e.g., snout
elongation). A simple ring wireframe connects consecutive landmarks for the 3D
plots.


In [ ]:
n_land   <- 12
n_dim    <- 3
n_per_taxon <- 6

# Base "skull" landmark template: points spaced around an ellipsoid.
theta_seq <- seq(0, 2 * pi, length.out = n_land + 1)[1:n_land]
base_template <- cbind(
  x = 10 * cos(theta_seq),
  y = 4  * sin(theta_seq),
  z = 3  * sin(2 * theta_seq)
)

# Deformation applied to the "Target" taxon's consensus (e.g. snout elongation
# concentrated on the first few landmarks).
target_template <- base_template
target_template[1:3, "x"] <- target_template[1:3, "x"] * 1.35

simulate_specimens <- function(template, n_specimens, noise_sd = 0.3) {
  array(
    apply(template, c(1, 2), function(v) v) [rep(seq_len(nrow(template)), n_specimens), ] +
      rnorm(nrow(template) * n_specimens * ncol(template), sd = noise_sd),
    dim = c(nrow(template), ncol(template), n_specimens)
  )
}
# (helper above intentionally simple; build the 3D array explicitly instead)
build_specimen_array <- function(template, n_specimens, noise_sd = 0.3) {
  arr <- array(NA_real_, dim = c(nrow(template), ncol(template), n_specimens))
  for (s in seq_len(n_specimens)) {
    arr[, , s] <- template + matrix(rnorm(length(template), sd = noise_sd), ncol = ncol(template))
  }
  arr
}

ancestor_specimens <- build_specimen_array(base_template, n_per_taxon)
target_specimens   <- build_specimen_array(target_template, n_per_taxon)

coords_data <- array(NA_real_, dim = c(n_land, n_dim, 2 * n_per_taxon))
coords_data[, , 1:n_per_taxon] <- ancestor_specimens
coords_data[, , (n_per_taxon + 1):(2 * n_per_taxon)] <- target_specimens

specimen_taxa <- c(rep("Ancestor", n_per_taxon), rep("Target", n_per_taxon))

# Simple ring wireframe: landmark i -> landmark i+1 (cyclic).
wireframe_links <- cbind(seq_len(n_land), c(2:n_land, 1))
wireframe_vector <- as.vector(t(wireframe_links))

morph_data <- list(
  all_skulls    = list(wireframe = wireframe_vector),
  coords_data   = coords_data,
  specimen_taxa = specimen_taxa,
  unique_taxa   = sort(unique(specimen_taxa)),
  n_land        = n_land,
  p_dim         = n_dim,
  n_vars        = n_land * n_dim
)

str(morph_data, max.level = 1)


## 4. Parameters

These replace the original Shiny sidebar inputs (`fileInput`, `numericInput`,
`sliderInput`, `selectInput`, `actionButton`, ...) with plain variables, so the
whole notebook is a deterministic function of the values set here.


In [ ]:
# --- tree topology ---
n_tips     <- 20     # number of tips in the simulated phylogeny
tree_scale <- 50     # total tree depth (Ma)

# --- time slicing ---
interval_by <- 2      # step size (Ma) between morphospace snapshots

# --- PCA retention ---
pc_mode           <- "threshold"  # "threshold" or "manual"
pc_variance       <- 0.95         # variance threshold if pc_mode == "threshold"
pc_count          <- 5            # PC count if pc_mode == "manual"

# --- landmark modules (blank = single "Global" module) ---
landmark_modules <- "Snout: 1,2,3\nBraincase: 4,5,6,7,8,9,10,11,12"

# --- node age overrides ("node_id = age" per line; blank = none) ---
node_age_overrides <- ""

# --- taxa and evolutionary model ---
ancestor_taxon <- "Ancestor"
target_taxon   <- "Target"
evol_model     <- "OU"    # "OU" or "BM"
alpha_val      <- 0.3     # OU pull strength
sigma_diag     <- 0.04    # baseline (within-PC) evolutionary rate
sigma_offdiag  <- 0.02    # within-module covariance


## 5. Run the pipeline

### 5.1 Shape space, modules, and Sigma

In [ ]:
pca_space <- build_pc_simulation_space(
  morph_data, ancestor_taxon, target_taxon,
  pc_mode = pc_mode, variance_threshold = pc_variance, requested_pcs = pc_count,
  taxa_limit = n_tips, module_text = landmark_modules,
  sigma_diag = sigma_diag, sigma_offdiag = sigma_offdiag
)

root_state    <- pca_space$root_state
target_state  <- pca_space$theta_state
sigma_matrix  <- pca_space$sigma_matrix

cat("Retained PCs:", pca_space$pca_model$retained_pcs,
    "of", length(pca_space$pca_model$explained_variance), "\n")
cat("Variance captured:",
    round(pca_space$pca_model$cumulative_variance[pca_space$pca_model$retained_pcs] * 100, 2), "%\n")
sigma_matrix


### 5.2 Tree topology and calibration

In [ ]:
base_tree <- pbtree(n = n_tips, scale = tree_scale)

calibrated <- build_calibrated_tree(base_tree, node_age_overrides)
tree <- calibrated$tree
tree_depth <- max(nodeHeights(tree))

intervals <- seq(1, tree_depth, by = interval_by)
if (length(intervals) == 0 || tail(intervals, 1) < tree_depth) {
  intervals <- sort(unique(c(intervals, tree_depth)))
}

cat("Tree depth:", round(tree_depth, 2), "Ma |", length(intervals), "time slices\n")

write.tree(tree, file = "simulated_tree.nwk")


### 5.3 Simulate + reconstruct evolutionary history (active model, plus BM & OU for comparison)

In [ ]:
active_history <- simulate_and_reconstruct_history(
  tree, root_state, sigma_matrix, target_state = target_state,
  model = evol_model, alpha_val = alpha_val, pc_labels = pca_space$sigma_info$pc_labels
)

bm_history <- simulate_and_reconstruct_history(
  tree, root_state, sigma_matrix, model = "BM",
  pc_labels = pca_space$sigma_info$pc_labels
)

ou_history <- simulate_and_reconstruct_history(
  tree, root_state, sigma_matrix, target_state = target_state, model = "OU",
  alpha_val = alpha_val, pc_labels = pca_space$sigma_info$pc_labels
)

# Back-project PC-space histories into landmark (skull) space.
back_project <- function(history_matrix) {
  reconstructed <- t(apply(history_matrix, 1, function(pc_scores) {
    as.numeric(pc_scores %*% t(pca_space$rotation) + pca_space$center)
  }))
  rownames(reconstructed) <- rownames(history_matrix)
  reconstructed
}

active_results <- back_project(active_history$full_history)
bm_results     <- back_project(bm_history$full_history)
ou_results     <- back_project(ou_history$full_history)


### 5.4 Time-slice snapshots and re-alignment

In [ ]:
active_snapshots_pc <- generate_all_snapshots(
  tree, intervals, active_history$full_history,
  alpha_val = if (evol_model == "OU") alpha_val else NULL, model = evol_model
)
bm_snapshots_pc <- generate_all_snapshots(tree, intervals, bm_history$full_history, model = "BM")
ou_snapshots_pc <- generate_all_snapshots(tree, intervals, ou_history$full_history,
                                           alpha_val = alpha_val, model = "OU")

back_project_snapshots <- function(snapshot_list) {
  lapply(snapshot_list, function(m) {
    r <- t(apply(m, 1, function(pc_scores) as.numeric(pc_scores %*% t(pca_space$rotation) + pca_space$center)))
    rownames(r) <- rownames(m)
    r
  })
}

active_snapshots <- back_project_snapshots(active_snapshots_pc)
bm_snapshots     <- back_project_snapshots(bm_snapshots_pc)
ou_snapshots     <- back_project_snapshots(ou_snapshots_pc)

active_snapshots_gpa <- perform_final_gpa_on_snapshots(active_snapshots, morph_data$n_land)
bm_snapshots_gpa     <- perform_final_gpa_on_snapshots(bm_snapshots, morph_data$n_land)
ou_snapshots_gpa     <- perform_final_gpa_on_snapshots(ou_snapshots, morph_data$n_land)


### 5.5 Mantel test: morphological vs. phylogenetic distance

In [ ]:
bm_tips  <- bm_results[seq_len(Ntip(tree)), , drop = FALSE]
ou_tips  <- ou_results[seq_len(Ntip(tree)), , drop = FALSE]

gpa_bm <- gpagen(reshape_to_array(bm_tips, morph_data$n_land), print.progress = FALSE)
gpa_ou <- gpagen(reshape_to_array(ou_tips, morph_data$n_land), print.progress = FALSE)

morph_dist_bm <- dist(two.d.array(gpa_bm$coords))
morph_dist_ou <- dist(two.d.array(gpa_ou$coords))
phylo_dist    <- as.dist(cophenetic(tree))

mantel_bm <- mantel(morph_dist_bm, phylo_dist, method = "pearson", permutations = 999)
mantel_ou <- mantel(morph_dist_ou, phylo_dist, method = "pearson", permutations = 999)

cat("BM Mantel test:\n"); print(mantel_bm)
cat("\nOU Mantel test:\n"); print(mantel_ou)


## 6. Visualizations

### 6.1 Tree with calibrated node ages and time-slice markers

In [ ]:
plot(tree, show.tip.label = TRUE, edge.width = 2)
abline(v = intervals, col = rgb(1, 0, 0, 0.2), lty = 2)
internal_node_ids <- (Ntip(tree) + 1):(Ntip(tree) + tree$Nnode)
nodelabels(text = internal_node_ids, frame = "circle", bg = "steelblue", cex = 0.8)
tiplabels(pch = 21, bg = "black", cex = 1)


### 6.2 Node age reference table

In [ ]:
cat(format_node_age_reference(tree, calibrated$node_ages))


### 6.3 Phylomorphospace (active model)

In [ ]:
plot_phylomorphospace(tree, active_history$full_history, morph_data$n_land,
                      title = paste(evol_model, "Phylomorphospace"),
                      target_config = if (evol_model == "OU") pca_space$target_vector else NULL)


### 6.4 Morphospace expansion through time (BM vs. OU)

In [ ]:
analyze_rigorous_morphospace(bm_snapshots, morph_data$n_land, title = "BM: Shapes under drift",
                             start_time_ma = min(intervals), end_time_ma = max(intervals))


In [ ]:
analyze_rigorous_morphospace(ou_snapshots, morph_data$n_land, title = "OU: Shapes under selection",
                             target_config = pca_space$target_vector,
                             start_time_ma = min(intervals), end_time_ma = max(intervals))


### 6.5 3D wireframe skulls (ancestor and target consensus)

`plotly` figures render as interactive HTML widgets. If widgets don't display
inline in your Jupyter setup, install/enable the `htmlwidgets`/`webshot` or
`IRdisplay` rendering path recommended in the `plotly` R package docs, or run
the notebook in JupyterLab with the `IRkernel` HTML repr enabled (default in
recent IRkernel versions).

In [ ]:
ancestor_consensus <- compute_taxon_consensus(morph_data, ancestor_taxon)
build_wireframe_plotly(ancestor_consensus, wireframe_links,
                        paste("Consensus wireframe:", ancestor_taxon), point_color = "steelblue")


In [ ]:
target_consensus <- compute_taxon_consensus(morph_data, target_taxon)
build_wireframe_plotly(target_consensus, wireframe_links,
                        paste("Consensus wireframe:", target_taxon), point_color = "firebrick")


### 6.6 Module / PC assignment summary

In [ ]:
module_lines <- vapply(names(pca_space$module_definitions), function(m) {
  paste0(m, ": ", paste(pca_space$module_definitions[[m]], collapse = ","))
}, character(1))
pc_lines <- paste0(pca_space$sigma_info$pc_labels, " -> ", pca_space$sigma_info$ordered_module_names)

cat("Modules:\n", paste(module_lines, collapse = "\n"), "\n\n",
    "Retained PC assignments:\n", paste(pc_lines, collapse = "\n"), sep = "")


## 7. Extending this workflow

- **Real data:** replace the "Load data" cell with
  `morph_data <- load_morphologika_data("your_file.txt")`.
- **Interactivity:** the original point-and-click tree/node inspector
  (`selection_data`, `output$skull_3d`) relied on Shiny's reactive event
  loop and is not reproduced here, since a static notebook has no click
  events. Instead, pick any node/tip/lineage of interest directly by row
  name in `active_results` or `active_snapshots[[...]]` and pass it to
  `build_wireframe_plotly()`, following the pattern in Section 6.5.
- **Parameter sweeps:** wrap Sections 5-6 in a loop over `evol_model`,
  `alpha_val`, or `sigma_offdiag` to compare regimes reproducibly.


## 8. Relation to FAIR data principles

**Findable.** The notebook records, in one place, exactly which inputs
(Morphologika file, tree topology, parameters) and which package versions
(`sessionInfo()`, Section 9) produced a given set of figures/tables, so a
specific run is identifiable, citable and reproducible. This repository is archived under Zenodo with the following DOI: [![DOI](https://zenodo.org/badge/1367178731.svg)](https://doi.org/10.5281/zenodo.22723974) that is its own, permenant identifier and is unique for this release. The data to run this script is available on GitHub and they don't come from a paper, so it isn't connected to a paper DOI (since this project was for a course, so the test data is random generated). Also, this project is findable by both humans and computers. By adding topics in the GitHub page, related to the project, it turns up in GitHub's topic browsing and search.

**Accessible.** Morphologika `.txt` is a plain-text format
readable by any text editor as well as by `geomorph::read.morphologika()`,
so the raw data has no access barrier. The notebook itself, and the
functions in it, are plain R that is easily installed and free. The DOI and GitHub page are public so everyone can have access.

**Interoperable.** Morphologika coordinates map directly onto the standard
`[landmarks, dimensions, specimens]` array `geomorph` and most other R
geometric-morphometrics packages expect, and the phylogeny is a standard
`ape`/`phytools` `phylo` object (Newick-compatible). PC scores, Procrustes
distances, and node ages are stored as plain numeric matrices/vectors with
informative row/column names (taxon names, node IDs, `PC1`, `PC2`, ...), which is what lets `geomorph`, `mvMORPH`,
`Rphylopars`, and `vegan` packages to operate on
the same objects easily. Also, the trees are exported in Newick format, that is the universal phylogenetics format for trees.

**Reusable.** Every analytical step (GPA, PCA, module definition, BM/OU
simulation, ancestral reconstruction, time-slicing, Mantel test) is a
documented, parameterized function and the
**Parameters** cell (Section 4) exposes every choice a re-user would need to
change (taxa, evolutionary model, rate/selection parameters, PC retention
rule, modules, node calibrations) without touching function internals. The
explicit LaTeX equations (see "Mathematical background", above) and markdown
comments describing *why* each step exists are
what let a re-user verify the analysis matches its stated
assumptions, which is the core of reusability. A MIT licence is added at the repository to make reuse terms
explicit. In addition, the *yaml* file creates the environment with the package versions used in the initial run, so the whole workflow can be run and produce the same results. In general, the structure of the notebook presents a reproducible pipeline.

## 9. Session info (for reproducibility)

In [ ]:
sessionInfo()
